# isThereEmotion — RAW TRACE에서 감정 관련 잠재 군집 찾기

> **연구 질문**  
> EmoNet의 RAW 뉴런 TRACE를 감정 정보 없이 분석했을 때 안정적인 내부 상태 군집이 발견되는가?  
> 또한 발견된 군집은 외부 감정 범주와 우연 이상의 연관성을 가지는가?

이 노트북 하나에서 **RAW JSON 전처리 → 라벨 블라인드 EDA → PCA → K-means → 계층적 군집화 → 군집 안정성 → 감정 라벨 사후 검증 → permutation test**까지 수행한다.

분석 라이브러리는 **NumPy와 pandas만 사용**한다. `json`, `pathlib`은 Python 표준 라이브러리이다.

## 분석 원칙

1. 군집 생성과 군집 수 선택에는 감정 `label`을 사용하지 않는다.
2. `top_emotions`, `dominant_global_signal`, latent `z_*`, 신경전달물질 모사값, style/LLM response 등 후단 감정 해석 결과는 feature에서 제외한다.
3. 군집 품질과 안정성을 먼저 확정한 뒤에만 감정 label을 공개한다.
4. 최종 결론은 “모델이 감정을 느낀다”가 아니라 **TRACE 내부에 감정과 체계적으로 연관된 잠재 구조가 존재하는지**에 한정한다.


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

SEED = 42
RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
RESULTS_DIR = Path("results")

EXPECTED_NODES = 256
PCA_VARIANCE_TARGET = 0.90
K_VALUES = range(2, 11)
N_INIT = 20
N_STABILITY = 30
N_PERMUTATIONS = 2000

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("RAW directory:", RAW_DIR.resolve())
print("raw_trace.json files:", len(list(RAW_DIR.rglob("raw_trace.json"))))


## 1. RAW TRACE 전처리 및 탐색

각 `raw_trace.json`은 여러 tick과 뉴런 상태가 중첩된 계층형 데이터이다. 이를 sample-level 수치 특징으로 바꾼다.

사용하는 RAW 정보는 `active_nodes`, `edges_fired`, `node_states.neuron_type`, `K`, `stim_vec`, tick 순서이다. 감정 label은 별도 metadata에 보관하고 **feature table과 물리적으로 분리**한다.

전처리에는 JSON 통합, 실패 TRACE 제거, 새로운 열 생성, `K`의 log 변환, 결측치 중앙값 처리, 저분산 열 제거, Z-score 표준화, PCA가 포함된다. 외부 plotting library를 사용할 수 없으므로 주요 분포는 NumPy 기반 ASCII histogram과 표로 확인한다.


In [ ]:
def safe_mean(values):
    return float(np.mean(values)) if len(values) else 0.0


def safe_std(values):
    return float(np.std(values)) if len(values) else 0.0


def slope(values):
    y = np.asarray(values, dtype=float)
    if len(y) < 2:
        return 0.0
    x = np.arange(len(y), dtype=float)
    x = x - x.mean()
    y = y - y.mean()
    denom = np.sum(x * x)
    return float(np.sum(x * y) / denom) if denom > 0 else 0.0


def time_to_fraction(values, fraction):
    if not values:
        return -1
    maximum = max(values)
    if maximum <= 0:
        return -1
    threshold = maximum * fraction
    for i, value in enumerate(values):
        if value >= threshold:
            return i
    return -1


def jaccard(a, b):
    union = a | b
    return len(a & b) / len(union) if union else 1.0


def signed_log1p(x):
    x = float(x)
    return np.sign(x) * np.log1p(abs(x))


def extract_trace(path, expected_nodes=EXPECTED_NODES):
    with path.open("r", encoding="utf-8") as f:
        raw = json.load(f)

    meta = raw.get("input_meta") or {}
    ticks = raw.get("ticks") or []

    metadata = {
        "sample_id": meta.get("sample_id") or path.parent.name,
        "label": meta.get("label"),
        "talk_id": meta.get("talk_id"),
        "persona_id": meta.get("persona_id"),
        "profile_id": meta.get("profile_id"),
        "source_file": path.as_posix(),
    }

    active_counts = []
    active_sets = []
    edge_counts = []
    unique_nodes = set()
    unique_edges = set()
    node_seen = np.zeros(expected_nodes, dtype=int)

    type_counts = {"excitatory": 0, "inhibitory": 0, "modulatory": 0}
    type_logk = {key: [] for key in type_counts}

    tick_logk_mean = []
    tick_stim_mean = []
    max_node_id = -1

    for tick in ticks:
        active = {int(x) for x in (tick.get("active_nodes") or [])}
        edges = tick.get("edges_fired") or []
        states = tick.get("node_states") or []

        active_counts.append(len(active))
        active_sets.append(active)
        edge_counts.append(len(edges))
        unique_nodes.update(active)

        for node_id in active:
            max_node_id = max(max_node_id, node_id)
            if 0 <= node_id < expected_nodes:
                node_seen[node_id] += 1

        for edge in edges:
            if isinstance(edge, (list, tuple)) and len(edge) >= 2:
                unique_edges.add((int(edge[0]), int(edge[1])))

        tick_logk = []
        tick_stim = []

        for state in states:
            node_id = state.get("node_id")
            if isinstance(node_id, int):
                max_node_id = max(max_node_id, node_id)

            neuron_type = state.get("neuron_type")
            if neuron_type in type_counts:
                type_counts[neuron_type] += 1

            k = state.get("K")
            if isinstance(k, (int, float)):
                lk = signed_log1p(k)
                tick_logk.append(lk)
                if neuron_type in type_logk:
                    type_logk[neuron_type].append(lk)

            stim = state.get("stim_vec")
            if isinstance(stim, list) and len(stim) == 4:
                try:
                    tick_stim.append(np.asarray(stim, dtype=float))
                except (TypeError, ValueError):
                    pass

        tick_logk_mean.append(safe_mean(tick_logk))
        if tick_stim:
            tick_stim_mean.append(np.mean(np.vstack(tick_stim), axis=0))
        else:
            tick_stim_mean.append(np.full(4, np.nan))

    inferred_nodes = max(expected_nodes, max_node_id + 1 if max_node_id >= 0 else 0)
    tick_count = len(ticks)
    active_total = float(np.sum(active_counts))
    edge_total = float(np.sum(edge_counts))

    consecutive_jaccard = [
        jaccard(active_sets[i - 1], active_sets[i])
        for i in range(1, len(active_sets))
    ]

    feature = {
        "sample_id": metadata["sample_id"],
        "ticks": tick_count,
        "network_size_inferred": inferred_nodes,
        "active_mean": safe_mean(active_counts),
        "active_std": safe_std(active_counts),
        "active_max": max(active_counts, default=0),
        "active_auc": active_total,
        "active_ratio_mean": safe_mean(active_counts) / inferred_nodes if inferred_nodes else 0.0,
        "active_slope_all": slope(active_counts),
        "active_slope_early10": slope(active_counts[:10]),
        "active_mean_early10": safe_mean(active_counts[:10]),
        "active_mean_late10": safe_mean(active_counts[-10:]),
        "time_to_50pct_active_max": time_to_fraction(active_counts, 0.50),
        "time_to_90pct_active_max": time_to_fraction(active_counts, 0.90),
        "zero_active_tick_ratio": np.mean(np.asarray(active_counts) == 0) if active_counts else 1.0,
        "unique_active_nodes": len(unique_nodes),
        "unique_active_node_ratio": len(unique_nodes) / inferred_nodes if inferred_nodes else 0.0,
        "edges_mean": safe_mean(edge_counts),
        "edges_std": safe_std(edge_counts),
        "edges_max": max(edge_counts, default=0),
        "edges_auc": edge_total,
        "edges_slope_all": slope(edge_counts),
        "edges_slope_early10": slope(edge_counts[:10]),
        "edge_per_active": edge_total / active_total if active_total else 0.0,
        "unique_edges": len(unique_edges),
        "edge_reuse_ratio": 1.0 - len(unique_edges) / edge_total if edge_total else 0.0,
        "active_set_jaccard_mean": safe_mean(consecutive_jaccard),
        "active_set_jaccard_std": safe_std(consecutive_jaccard),
        "logK_mean": safe_mean(tick_logk_mean),
        "logK_std": safe_std(tick_logk_mean),
        "logK_slope": slope(tick_logk_mean),
    }

    total_type_states = sum(type_counts.values())
    for neuron_type, short in [
        ("excitatory", "exc"),
        ("inhibitory", "inh"),
        ("modulatory", "mod"),
    ]:
        feature[f"{short}_state_ratio"] = (
            type_counts[neuron_type] / total_type_states if total_type_states else 0.0
        )
        feature[f"{short}_logK_mean"] = safe_mean(type_logk[neuron_type])
        feature[f"{short}_logK_std"] = safe_std(type_logk[neuron_type])

    stim_matrix = np.vstack(tick_stim_mean) if tick_stim_mean else np.empty((0, 4))
    for j in range(4):
        values = stim_matrix[:, j] if len(stim_matrix) else np.asarray([])
        valid = values[np.isfinite(values)] if len(values) else values
        feature[f"stim{j}_mean"] = safe_mean(valid)
        feature[f"stim{j}_std"] = safe_std(valid)
        feature[f"stim{j}_slope"] = slope(valid)

    denom = max(tick_count, 1)
    rates = node_seen / denom
    feature["node_activation_rate_mean"] = float(rates.mean())
    feature["node_activation_rate_std"] = float(rates.std())
    feature["variable_node_ratio"] = float(np.mean(rates * (1 - rates) > 0.05))
    feature["high_persistence_node_ratio"] = float(np.mean(rates >= 0.90))

    p = np.clip(rates, 1e-12, 1 - 1e-12)
    feature["node_binary_entropy_mean"] = float(
        np.mean(-(p * np.log(p) + (1 - p) * np.log(1 - p)))
    )

    for node_id, rate in enumerate(rates):
        feature[f"node_freq_{node_id:03d}"] = float(rate)

    return metadata, feature


raw_files = sorted(RAW_DIR.rglob("raw_trace.json"))
metadata_rows = []
feature_rows = []

for i, path in enumerate(raw_files, start=1):
    meta, feature = extract_trace(path)
    metadata_rows.append(meta)
    feature_rows.append(feature)
    if i % 20 == 0 or i == len(raw_files):
        print(f"processed {i:3d}/{len(raw_files)}")

metadata = pd.DataFrame(metadata_rows)
features = pd.DataFrame(feature_rows)

metadata.to_csv(PROCESSED_DIR / "metadata_labels.csv", index=False, encoding="utf-8-sig")
features.to_csv(PROCESSED_DIR / "trace_features.csv", index=False, encoding="utf-8-sig")

print("metadata shape:", metadata.shape)
print("feature shape :", features.shape)

# label과 무관한 품질 기준으로 실패/빈 TRACE 제거
qc_mask = (features["ticks"] >= 3) & (features["unique_active_nodes"] > 0)
removed = features.loc[~qc_mask, ["sample_id", "ticks", "unique_active_nodes"]]
blind_features = features.loc[qc_mask].reset_index(drop=True)

print("usable samples :", len(blind_features))
print("removed samples:", len(removed))
if len(removed):
    print(removed.to_string(index=False))

feature_cols = [c for c in blind_features.columns if c != "sample_id"]
X_df = blind_features[feature_cols].apply(pd.to_numeric, errors="coerce")
X_df = X_df.apply(lambda s: s.fillna(s.median()))

# 거의 변하지 않는 feature는 거리 계산에 의미가 없으므로 제거
std = X_df.std(ddof=0)
keep_cols = std[std > 1e-12].index.tolist()
X_df = X_df[keep_cols]

mean = X_df.mean()
std = X_df.std(ddof=0).replace(0, 1.0)
X_z = ((X_df - mean) / std).to_numpy(dtype=float)

print("features before low-variance removal:", len(feature_cols))
print("features after  low-variance removal:", len(keep_cols))
print("standardized matrix:", X_z.shape)


def pca_numpy(X, variance_target=0.90):
    X = np.asarray(X, dtype=float)
    X_centered = X - X.mean(axis=0)
    U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)
    eigenvalues = (S ** 2) / max(len(X) - 1, 1)
    explained = eigenvalues / eigenvalues.sum()
    cumulative = np.cumsum(explained)
    n_components = int(np.searchsorted(cumulative, variance_target) + 1)
    components = Vt[:n_components]
    scores = X_centered @ components.T
    return scores, components, explained, cumulative


X_pca, pca_components, explained, cumulative = pca_numpy(X_z, PCA_VARIANCE_TARGET)
pca_summary = pd.DataFrame({
    "PC": np.arange(1, len(explained) + 1),
    "explained_variance_ratio": explained,
    "cumulative_ratio": cumulative,
})

print("PCA components used:", X_pca.shape[1])
print("cumulative explained variance:", cumulative[X_pca.shape[1] - 1])


def ascii_hist(values, bins=10, width=36):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    counts, edges = np.histogram(values, bins=bins)
    max_count = max(counts.max(), 1)
    lines = []
    for count, left, right in zip(counts, edges[:-1], edges[1:]):
        bar = "█" * int(round(width * count / max_count))
        lines.append(f"{left:9.3f} ~ {right:9.3f} | {bar} {count}")
    return "\n".join(lines)


for col in ["active_ratio_mean", "edges_mean", "logK_mean", "active_set_jaccard_mean"]:
    if col in blind_features:
        print("\n[" + col + "]")
        print(ascii_hist(blind_features[col], bins=10))

pca_summary.head(15)


## 2. 감정 label을 보지 않는 군집 분석

서로 다른 원리의 두 알고리즘을 비교한다.

- **K-means clustering**: 중심점과의 거리 기반
- **Average-linkage hierarchical clustering**: 가까운 sample/cluster를 순차적으로 병합

`k=2~10`을 모두 검사하고, 감정 label이 아니라 **Silhouette score**만으로 군집 수를 비교한다. 두 알고리즘의 평균 Silhouette가 가장 높은 `K*`를 고정한 뒤, ARI로 알고리즘 간 재현성과 seed/작은 perturbation에 대한 안정성을 확인한다. feature-wise shuffle negative control도 함께 수행한다.


In [ ]:
def pairwise_euclidean(X):
    X = np.asarray(X, dtype=float)
    sq = np.sum(X * X, axis=1, keepdims=True)
    d2 = np.maximum(sq + sq.T - 2 * X @ X.T, 0.0)
    return np.sqrt(d2)


def kmeans_pp_init(X, k, rng):
    n = len(X)
    centers_idx = [int(rng.integers(n))]
    closest_d2 = np.sum((X - X[centers_idx[0]]) ** 2, axis=1)

    for _ in range(1, k):
        total = closest_d2.sum()
        if total <= 1e-15:
            candidates = [i for i in range(n) if i not in centers_idx]
            idx = int(rng.choice(candidates))
        else:
            idx = int(rng.choice(n, p=closest_d2 / total))

        centers_idx.append(idx)
        d2 = np.sum((X - X[idx]) ** 2, axis=1)
        closest_d2 = np.minimum(closest_d2, d2)

    return X[centers_idx].copy()


def kmeans_once(X, k, seed=0, max_iter=300, tol=1e-8):
    X = np.asarray(X, dtype=float)
    rng = np.random.default_rng(seed)
    centers = kmeans_pp_init(X, k, rng)
    labels = np.full(len(X), -1, dtype=int)

    for _ in range(max_iter):
        d2 = np.sum((X[:, None, :] - centers[None, :, :]) ** 2, axis=2)
        new_labels = np.argmin(d2, axis=1)

        new_centers = np.empty_like(centers)
        min_d2 = d2[np.arange(len(X)), new_labels]
        used = set()

        for c in range(k):
            members = X[new_labels == c]
            if len(members):
                new_centers[c] = members.mean(axis=0)
            else:
                order = np.argsort(min_d2)[::-1]
                idx = next(int(i) for i in order if int(i) not in used)
                used.add(idx)
                new_centers[c] = X[idx]

        shift = np.sqrt(np.sum((new_centers - centers) ** 2, axis=1)).max()
        centers = new_centers

        if np.array_equal(new_labels, labels) or shift < tol:
            labels = new_labels
            break
        labels = new_labels

    inertia = float(np.sum((X - centers[labels]) ** 2))
    return labels, centers, inertia


def kmeans_numpy(X, k, n_init=20, seed=42):
    master = np.random.default_rng(seed)
    best = None

    for s in master.integers(0, 2**31 - 1, size=n_init):
        result = kmeans_once(X, k, int(s))
        if best is None or result[2] < best[2]:
            best = result

    return best


def silhouette_score_numpy(X, labels, distance_matrix=None):
    X = np.asarray(X, dtype=float)
    labels = np.asarray(labels)
    D = pairwise_euclidean(X) if distance_matrix is None else distance_matrix
    clusters = np.unique(labels)

    if len(clusters) < 2 or len(clusters) >= len(labels):
        return np.nan

    scores = np.zeros(len(labels), dtype=float)

    for i in range(len(labels)):
        same = np.where(labels == labels[i])[0]
        same = same[same != i]

        if len(same) == 0:
            scores[i] = 0.0
            continue

        a = D[i, same].mean()
        b = min(D[i, labels == c].mean() for c in clusters if c != labels[i])
        scores[i] = (b - a) / max(a, b) if max(a, b) > 0 else 0.0

    return float(scores.mean())


def hierarchical_average_labels_by_k(X, ks):
    X = np.asarray(X, dtype=float)
    n = len(X)
    wanted = set(ks)
    D0 = pairwise_euclidean(X)

    # 병합마다 새 cluster ID가 필요하므로 최대 2n-1개의 노드를 사용한다.
    max_nodes = 2 * n - 1
    D = np.full((max_nodes, max_nodes), np.inf, dtype=float)
    D[:n, :n] = D0
    np.fill_diagonal(D, np.inf)

    sizes = np.zeros(max_nodes, dtype=int)
    sizes[:n] = 1
    members = {i: [i] for i in range(n)}
    active = list(range(n))
    labels_by_k = {}
    next_id = n

    while len(active) > 1:
        idx = np.asarray(active, dtype=int)
        sub = D[np.ix_(idx, idx)]
        ai, bi = np.unravel_index(np.argmin(sub), sub.shape)
        a, b = int(idx[ai]), int(idx[bi])

        new = next_id
        next_id += 1
        sizes[new] = sizes[a] + sizes[b]
        members[new] = members[a] + members[b]

        # Average linkage: 두 기존 cluster의 크기로 가중한 평균 거리.
        for c in active:
            if c in (a, b):
                continue
            d = (sizes[a] * D[a, c] + sizes[b] * D[b, c]) / sizes[new]
            D[new, c] = D[c, new] = d

        active = [c for c in active if c not in (a, b)] + [new]

        if len(active) in wanted:
            labels = np.empty(n, dtype=int)
            for label, cluster_id in enumerate(active):
                labels[members[cluster_id]] = label
            labels_by_k[len(active)] = labels.copy()

        if len(active) <= min(wanted):
            break

    return labels_by_k


D = pairwise_euclidean(X_pca)

kmeans_results = {}
kmeans_rows = []

for k in K_VALUES:
    labels, centers, inertia = kmeans_numpy(X_pca, k, n_init=N_INIT, seed=SEED)
    sil = silhouette_score_numpy(X_pca, labels, D)
    kmeans_results[k] = labels
    kmeans_rows.append({
        "k": k,
        "kmeans_silhouette": sil,
        "kmeans_inertia": inertia,
        "kmeans_min_cluster_size": int(pd.Series(labels).value_counts().min()),
    })

hierarchical_results = hierarchical_average_labels_by_k(X_pca, K_VALUES)
hier_rows = []

for k in K_VALUES:
    labels = hierarchical_results[k]
    sil = silhouette_score_numpy(X_pca, labels, D)
    hier_rows.append({
        "k": k,
        "hierarchical_silhouette": sil,
        "hierarchical_min_cluster_size": int(pd.Series(labels).value_counts().min()),
    })

comparison = pd.DataFrame(kmeans_rows).merge(pd.DataFrame(hier_rows), on="k")
comparison["mean_silhouette"] = comparison[
    ["kmeans_silhouette", "hierarchical_silhouette"]
].mean(axis=1)

# 감정 label을 보지 않고 두 알고리즘 평균 silhouette가 가장 높은 K를 고정한다.
K_STAR = int(comparison.loc[comparison["mean_silhouette"].idxmax(), "k"])
comparison.to_csv(RESULTS_DIR / "blind_cluster_comparison.csv", index=False, encoding="utf-8-sig")

print("K* selected without emotion labels:", K_STAR)
print(comparison.to_string(index=False))


def adjusted_rand_index(a, b):
    a = np.asarray(a)
    b = np.asarray(b)
    _, ai = np.unique(a, return_inverse=True)
    _, bi = np.unique(b, return_inverse=True)

    contingency = np.zeros((ai.max() + 1, bi.max() + 1), dtype=int)
    np.add.at(contingency, (ai, bi), 1)

    def comb2(x):
        x = np.asarray(x, dtype=float)
        return x * (x - 1) / 2

    sum_ij = comb2(contingency).sum()
    sum_a = comb2(contingency.sum(axis=1)).sum()
    sum_b = comb2(contingency.sum(axis=0)).sum()
    total = comb2(len(a))
    expected = sum_a * sum_b / total if total else 0.0
    maximum = 0.5 * (sum_a + sum_b)
    denom = maximum - expected
    return float((sum_ij - expected) / denom) if abs(denom) > 1e-15 else 1.0


kmeans_labels = kmeans_results[K_STAR]
hier_labels = hierarchical_results[K_STAR]
method_ari = adjusted_rand_index(kmeans_labels, hier_labels)

cluster_sizes = pd.DataFrame({
    "kmeans": pd.Series(kmeans_labels).value_counts().sort_index(),
    "hierarchical": pd.Series(hier_labels).value_counts().sort_index(),
})

print("\nK-means ↔ Hierarchical ARI:", method_ari)
print(cluster_sizes.to_string())

# K-means 초기 중심 변화에 대한 안정성.
seed_ari = []
for s in range(N_STABILITY):
    labels_s, _, _ = kmeans_once(X_pca, K_STAR, seed=s)
    seed_ari.append(adjusted_rand_index(kmeans_labels, labels_s))

# PCA 좌표에 작은 5% 잡음을 넣어도 같은 군집이 유지되는지 확인.
rng = np.random.default_rng(SEED)
noise_ari = []
scale = X_pca.std(axis=0)
scale[scale == 0] = 1.0

for _ in range(N_STABILITY):
    X_noisy = X_pca + rng.normal(0.0, 0.05, size=X_pca.shape) * scale
    labels_noisy, _, _ = kmeans_numpy(
        X_noisy,
        K_STAR,
        n_init=10,
        seed=int(rng.integers(0, 2**31 - 1)),
    )
    noise_ari.append(adjusted_rand_index(kmeans_labels, labels_noisy))

stability = pd.DataFrame({
    "test": ["seed_change", "5pct_feature_noise"],
    "mean_ARI": [np.mean(seed_ari), np.mean(noise_ari)],
    "median_ARI": [np.median(seed_ari), np.median(noise_ari)],
    "min_ARI": [np.min(seed_ari), np.min(noise_ari)],
    "max_ARI": [np.max(seed_ari), np.max(noise_ari)],
})
stability.to_csv(RESULTS_DIR / "cluster_stability.csv", index=False, encoding="utf-8-sig")

print("\nStability")
print(stability.to_string(index=False))


def shuffled_feature_control(X, k, observed_silhouette, repeats=100, seed=42):
    X = np.asarray(X, dtype=float)
    rng = np.random.default_rng(seed)
    null_scores = []

    for _ in range(repeats):
        X_null = X.copy()
        # 각 PCA 축의 주변분포는 유지하면서 sample별 축 조합을 깨뜨린다.
        for j in range(X_null.shape[1]):
            rng.shuffle(X_null[:, j])

        labels_null, _, _ = kmeans_numpy(
            X_null,
            k,
            n_init=5,
            seed=int(rng.integers(0, 2**31 - 1)),
        )
        null_scores.append(silhouette_score_numpy(X_null, labels_null))

    null_scores = np.asarray(null_scores)
    p = (1 + np.sum(null_scores >= observed_silhouette)) / (len(null_scores) + 1)
    return null_scores, float(p)


observed_silhouette = silhouette_score_numpy(X_pca, kmeans_labels, D)
control_silhouette, control_p = shuffled_feature_control(
    X_pca,
    K_STAR,
    observed_silhouette,
    repeats=100,
    seed=SEED,
)

print("\nNegative control")
print("observed silhouette:", observed_silhouette)
print("control mean       :", control_silhouette.mean())
print("control max        :", control_silhouette.max())
print("empirical p-value  :", control_p)


## 3. 🔓 감정 label 봉인 해제 — 사후 검증

**여기까지의 PCA, K*, 군집 결과는 감정 label을 사용하지 않고 먼저 확정한다.**

이제 처음으로 외부 감정 label을 공개한다.

- cluster ↔ emotion 연관성: **Normalized Mutual Information (NMI)**
- 우연 여부: **label permutation test**
- 특정 clustering 알고리즘에 의존하지 않는 보강 증거: **same-emotion distance vs different-emotion distance + permutation test**

현재 E-code는 세부 범주가 많고 희소한 class가 있으므로 NMI 절대값만 해석하지 않고, 같은 label 빈도를 유지한 permutation null distribution과 비교한다.


In [ ]:
# 여기부터 감정 label 봉인 해제: K*, PCA, 군집 결과를 먼저 확정한 뒤에만 실행한다.
valid_ids = blind_features["sample_id"].tolist()
validation_meta = (
    metadata
    .set_index("sample_id")
    .loc[valid_ids]
    .reset_index()
)

emotion = validation_meta["label"].astype(str).to_numpy()

print("emotion classes:", pd.Series(emotion).nunique())
print("samples        :", len(emotion))
print("\nlabel counts")
print(pd.Series(emotion).value_counts().to_string())


def nmi_score(a, b):
    a = np.asarray(a)
    b = np.asarray(b)
    _, ai = np.unique(a, return_inverse=True)
    _, bi = np.unique(b, return_inverse=True)

    contingency = np.zeros((ai.max() + 1, bi.max() + 1), dtype=float)
    np.add.at(contingency, (ai, bi), 1.0)

    n = contingency.sum()
    pxy = contingency / n
    px = pxy.sum(axis=1, keepdims=True)
    py = pxy.sum(axis=0, keepdims=True)
    expected = px @ py
    nonzero = pxy > 0

    mutual_information = np.sum(
        pxy[nonzero] * np.log(pxy[nonzero] / expected[nonzero])
    )

    px1 = px.ravel()
    py1 = py.ravel()
    hx = -np.sum(px1[px1 > 0] * np.log(px1[px1 > 0]))
    hy = -np.sum(py1[py1 > 0] * np.log(py1[py1 > 0]))
    denom = np.sqrt(hx * hy)
    return float(mutual_information / denom) if denom > 0 else 0.0


def permutation_test_nmi(cluster_labels, emotion_labels, repeats=2000, seed=42):
    observed = nmi_score(cluster_labels, emotion_labels)
    rng = np.random.default_rng(seed)
    null = np.empty(repeats, dtype=float)

    for i in range(repeats):
        shuffled = rng.permutation(emotion_labels)
        null[i] = nmi_score(cluster_labels, shuffled)

    p = (1 + np.sum(null >= observed)) / (repeats + 1)
    return observed, null, float(p)


nmi_observed, nmi_null, nmi_p = permutation_test_nmi(
    kmeans_labels,
    emotion,
    repeats=N_PERMUTATIONS,
    seed=SEED,
)

print("\nCluster ↔ emotion")
print("observed NMI :", nmi_observed)
print("null mean    :", nmi_null.mean())
print("null 95%     :", np.quantile(nmi_null, 0.95))
print("p-value      :", nmi_p)

cluster_emotion_table = pd.crosstab(
    pd.Series(kmeans_labels, name="cluster"),
    pd.Series(emotion, name="emotion"),
)
print("\ncluster × emotion")
print(cluster_emotion_table.to_string())


def emotion_distance_test(X, emotion_labels, repeats=2000, seed=42):
    X = np.asarray(X, dtype=float)
    labels = np.asarray(emotion_labels)
    D = pairwise_euclidean(X)
    upper = np.triu_indices(len(X), 1)
    pair_distance = D[upper]
    same = labels[upper[0]] == labels[upper[1]]

    if not same.any() or same.all():
        raise ValueError("same/different emotion pair comparison is not possible.")

    same_mean = float(pair_distance[same].mean())
    different_mean = float(pair_distance[~same].mean())
    # 양수일수록 같은 감정 sample끼리 더 가깝다.
    observed_gap = different_mean - same_mean

    rng = np.random.default_rng(seed)
    null = np.empty(repeats, dtype=float)

    for i in range(repeats):
        shuffled = rng.permutation(labels)
        shuffled_same = shuffled[upper[0]] == shuffled[upper[1]]
        null[i] = (
            pair_distance[~shuffled_same].mean()
            - pair_distance[shuffled_same].mean()
        )

    p = (1 + np.sum(null >= observed_gap)) / (repeats + 1)
    return same_mean, different_mean, observed_gap, null, float(p)


same_dist, diff_dist, distance_gap, distance_null, distance_p = emotion_distance_test(
    X_pca,
    emotion,
    repeats=N_PERMUTATIONS,
    seed=SEED,
)

print("\nSame emotion vs different emotion distance")
print("same-emotion mean distance     :", same_dist)
print("different-emotion mean distance:", diff_dist)
print("distance gap (different - same):", distance_gap)
print("null 95%                       :", np.quantile(distance_null, 0.95))
print("p-value                        :", distance_p)

assignments = pd.DataFrame({
    "sample_id": blind_features["sample_id"],
    "cluster_kmeans": kmeans_labels,
    "cluster_hierarchical": hier_labels,
    "emotion_label_after_unblinding": emotion,
})

summary = pd.DataFrame([
    {"metric": "selected_k_blind", "value": K_STAR, "p_value": np.nan},
    {"metric": "kmeans_silhouette", "value": observed_silhouette, "p_value": control_p},
    {"metric": "kmeans_vs_hierarchical_ARI", "value": method_ari, "p_value": np.nan},
    {"metric": "seed_stability_median_ARI", "value": np.median(seed_ari), "p_value": np.nan},
    {"metric": "noise_stability_median_ARI", "value": np.median(noise_ari), "p_value": np.nan},
    {"metric": "cluster_emotion_NMI", "value": nmi_observed, "p_value": nmi_p},
    {"metric": "emotion_distance_gap", "value": distance_gap, "p_value": distance_p},
])

assignments.to_csv(
    RESULTS_DIR / "cluster_assignments.csv",
    index=False,
    encoding="utf-8-sig",
)
summary.to_csv(
    RESULTS_DIR / "validation_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

print("\nSaved results")
print("-", RESULTS_DIR / "blind_cluster_comparison.csv")
print("-", RESULTS_DIR / "cluster_stability.csv")
print("-", RESULTS_DIR / "cluster_assignments.csv")
print("-", RESULTS_DIR / "validation_summary.csv")

summary


## 4. 결과 해석 기준

이 프로젝트에서는 하나의 숫자만으로 “감정이 있다”고 결론내리지 않는다.

- **뚜렷한 군집 자체가 없음** → 현재 TRACE에서 안정적인 잠재 상태 구조의 증거가 부족하다.
- **안정적인 군집은 있으나 emotion permutation test가 유의하지 않음** → 내부 구조는 있으나 감정 구조라고 부를 증거는 부족하다.
- **군집이 안정적이고 서로 다른 알고리즘에서도 재현되며 emotion과의 연관성이 permutation null보다 강함** → TRACE 내부에 **감정과 체계적으로 연관된 잠재 구조가 존재한다는 증거**가 된다.
- **same-emotion distance까지 더 작고 permutation test에서도 유의함** → 특정 clustering 알고리즘에만 의존하지 않는 독립적인 보강 증거가 된다.

어떤 경우에도 이 결과만으로 EmoNet이 주관적으로 감정을 “느낀다”고 주장하지 않는다.

## 수행평가 체크리스트

| 평가 항목 | 노트북에서 수행하는 내용 |
|---|---|
| 문제 정의 | RAW TRACE에서 감정 관련 잠재 군집이 발견되는가? |
| 데이터 수집 | EmoNet에서 생성한 독립 RAW TRACE 120개 |
| 전처리 | JSON 통합, 실패 TRACE 제거, 새 feature 생성, log 변환, 결측 처리, 저분산 제거, 표준화 |
| 탐색 | feature 분포, PCA 설명분산 |
| 알고리즘 1 | K-means clustering |
| 알고리즘 2 | Average-linkage hierarchical clustering |
| 선정 이유 | 서로 다른 원리에서도 내부 구조가 재현되는지 확인 |
| 평가 지표 | Silhouette score, ARI 기반 안정성 |
| 추가 검증 | NMI + permutation test, same/different emotion distance test |
| 활용 | TRACE 내부에 감정과 관련된 잠재 구조가 존재하는지 데이터로 판단 |

### 핵심 한 문장

**감정 label을 보지 않고 발견한 안정적인 TRACE 군집이, label을 공개한 뒤에도 우연 이상의 감정 연관성을 보이는지를 검증한다.**
